<a href="https://colab.research.google.com/github/Susichithra/Project_1/blob/main/Earthquake_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
!pip install pandas

!pip install pymysql sqlalchemy

!pip install streamlit

In [63]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import requests
from sqlalchemy import create_engine
import time
time.sleep(0.5)
import sqlite3
import streamlit as st

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
start_year = datetime.now().year - 5   # last 5 years
end_year = datetime.now().year -1

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f" Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f" JSON error for {start_date}: {e}")
            continue


        for f in data.get ("features",[]):
            p = f["properties"]
            geom = f.get("geometry")
            g = geom["coordinates"] if geom else [None, None, None]


            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "magError": p.get("magError"),
                "depthError": p.get("depthError"),
                "magNst": p.get("magNst"),
                "locationSource": p.get("locationSource"),
                "magSource": p.get("magSource"),
                "types": p.get("types"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "type": p.get("type")

                })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())

Rows: 104441
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...    gap  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   42.0   
1                                        Fiji region  reviewed  ...   64.0   
2                    103 km SW of Basco, Philippines  r

#Data Cleaning

In [64]:
len(df)

104441

In [65]:

df.isnull().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [66]:
# Fill missing numerical values with median
num_cols = ['nst', 'dmin', 'rms', 'gap', 'magError',
            'depthError', 'magNst']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical columns with 'Unknown'
cat_cols = ['locationSource', 'magSource']

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/tmp/ipykernel_677/4242298171.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].median())
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/tmp/ipykernel_677/4242298171.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = d

In [67]:
print(df.isnull().sum())

id                     0
time                   0
updated                0
latitude               0
longitude              0
depth_km               0
mag                    0
magType                0
place                  0
status                 0
tsunami                0
sig                    0
net                    0
nst                    0
dmin                   0
rms                    0
gap                    0
magError          104441
depthError        104441
magNst            104441
locationSource         0
magSource              0
types                  0
ids                    0
sources                0
type                   0
dtype: int64


In [68]:
df.drop_duplicates(inplace=True)

In [69]:
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,gap,magError,depthError,magNst,locationSource,magSource,types,ids,sources,type
0,us6000ddi8,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040,-31.7493,-68.9337,17.27,4.7,mwr,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,...,42.0,NaN,NaN,NaN,Unknown,Unknown,",dyfi,moment-tensor,origin,phase-data,",",us6000ddi8,",",us,",earthquake
1,us6000dev6,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040,-15.4902,-177.2052,426.71,4.1,mb,Fiji region,reviewed,...,64.0,NaN,NaN,NaN,Unknown,Unknown,",origin,phase-data,",",us6000dev6,",",us,",earthquake
2,us6000dev5,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040,19.7529,121.3159,46.73,4.7,mb,"103 km SW of Basco, Philippines",reviewed,...,106.0,NaN,NaN,NaN,Unknown,Unknown,",origin,phase-data,",",us6000dev5,",",us,",earthquake
3,us6000ddhs,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040,28.1524,57.2570,10.00,4.9,mb,"114 km N of M?n?b, Iran",reviewed,...,71.0,NaN,NaN,NaN,Unknown,Unknown,",origin,phase-data,",",us6000ddhs,",",us,",earthquake
4,us6000dev4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040,71.3212,-3.7578,10.00,4.0,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,...,65.0,NaN,NaN,NaN,Unknown,Unknown,",origin,phase-data,",",us6000dev4,",",us,",earthquake


In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104441 entries, 0 to 104440
Data columns (total 26 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id              104441 non-null  object        
 1   time            104441 non-null  datetime64[ns]
 2   updated         104441 non-null  datetime64[ns]
 3   latitude        104441 non-null  float64       
 4   longitude       104441 non-null  float64       
 5   depth_km        104441 non-null  float64       
 6   mag             104441 non-null  float64       
 7   magType         104441 non-null  object        
 8   place           104441 non-null  object        
 9   status          104441 non-null  object        
 10  tsunami         104441 non-null  int64         
 11  sig             104441 non-null  int64         
 12  net             104441 non-null  object        
 13  nst             104441 non-null  float64       
 14  dmin            104441 non-null  flo

In [71]:
# Convert Timestamps Columns
df["time"] = pd.to_datetime(df["time"])
df["updated"] = pd.to_datetime(df["updated"])

In [72]:
df[["time","updated"]]

,time,updated
0,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040
1,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040
2,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040
3,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040
4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040
...,...,...
104436,2025-12-01 01:51:06.974,2026-02-25 16:18:40.040
104437,2025-12-01 01:41:43.796,2026-02-25 16:18:40.040
104438,2025-12-01 01:37:57.834,2026-02-25 16:18:40.040
104439,2025-12-01 01:25:46.725,2026-02-25 16:18:40.040


In [73]:
# Numerical Column Id
numeric_cols = [
    "mag",
    "depth_km",
    "nst",
    "dmin",
    "rms",
    "gap",
    "magError",
    "depthError",
    "magNst",
    "sig"
]

In [74]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [75]:
# Clean Text

text_cols = [
    "magType",
    "place",
    "status",
    "net",
    "locationSource",
    "magSource",
    "types",
    "ids",
    "sources",
    "type"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()


In [76]:
df["status"] = df["status"].str.lower()
df["type"] = df["type"].str.lower()

In [77]:
# Extract Country

def extract_country(place):

    if pd.isnull(place):
        return "Unknown"

    parts = place.split(",")

    if len(parts) > 1:
        return parts[-1].strip()

    return "Unknown"

df["country"] = df["place"].apply(extract_country)

In [78]:
df[["place", "country"]].head(10)

,place,country
0,"29 km SW of Villa Basilio Nievas, Argentina",Argentina
1,Fiji region,Unknown
2,"103 km SW of Basco, Philippines",Philippines
3,"114 km N of M?n?b, Iran",Iran
4,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",Svalbard and Jan Mayen
5,"76 km NNE of Luquillo, Puerto Rico",Puerto Rico
6,"78 km NNE of Luquillo, Puerto Rico",Puerto Rico
7,"82 km N of Culebra, Puerto Rico",Puerto Rico
8,"99 km SE of Pondaguitan, Philippines",Philippines
9,"69 km SE of Pondaguitan, Philippines",Philippines


In [79]:
# Create Drived Column

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["hour"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.day_name()

In [80]:
# Depth Category

def depth_category(depth):

    if depth < 70:
        return "Shallow"

    elif depth < 300:
        return "Intermediate"

    else:
        return "Deep"

df["depth_category"] = df["depth_km"].apply(depth_category)

In [81]:
# Magnitude Category

def magnitude_category(mag):

    if mag >= 8:
        return "Great"

    elif mag >= 7:
        return "Major"

    elif mag >= 6:
        return "Strong"

    elif mag >= 5:
        return "Moderate"

    else:
        return "Light"

df["magnitude_category"] = df["mag"].apply(magnitude_category)

In [82]:
# Flag

df["strong_eq"] = np.where(df["mag"] >= 6.5, 1, 0)

In [83]:
#df.info()
df.describe()
#df.head()

,time,updated,latitude,longitude,depth_km,mag,tsunami,sig,nst,dmin,rms,gap,magError,depthError,magNst,year,month,day,hour,strong_eq
count,104441,104441,104441.000000,104441.000000,104441.000000,104441.000000,104441.000000,104441.000000,104441.000000,104441.00000,104441.000000,104441.000000,0.0,0.0,0.0,104441.000000,104441.000000,104441.000000,104441.000000,104441.000000
mean,2023-07-03 07:21:30.440105472,2023-10-25 01:38:32.055430912,11.798143,6.875245,72.059931,4.260394,0.005984,288.047261,41.741194,3.14765,0.654422,121.494065,NaN,NaN,NaN,2023.002250,6.520514,15.505051,11.488668,0.002039
min,2021-01-01 00:14:07.580000,2021-01-01 10:15:32.601000,-73.220400,-179.999700,-3.740000,3.000000,0.000000,138.000000,0.000000,0.00000,0.000000,8.000000,NaN,NaN,NaN,2021.000000,1.000000,1.000000,0.000000,0.000000
25%,2022-03-13 19:00:46.580000,2022-07-05 03:41:38.040000,-14.986700,-113.801900,10.000000,4.000000,0.000000,259.000000,24.000000,0.86900,0.490000,76.000000,NaN,NaN,NaN,2022.000000,3.000000,8.000000,5.000000,0.000000
50%,2023-06-26 01:33:48.232999936,2023-12-02 23:01:08.040000,14.528000,22.438600,21.100000,4.300000,0.000000,285.000000,32.000000,1.81600,0.650000,112.000000,NaN,NaN,NaN,2023.000000,7.000000,15.000000,11.000000,0.000000
75%,2024-10-31 20:20:40.436000,2025-02-27 17:30:48.040000,38.655500,130.386100,71.320000,4.600000,0.000000,326.000000,45.000000,3.59700,0.820000,153.000000,NaN,NaN,NaN,2024.000000,9.000000,23.000000,18.000000,0.000000
max,2025-12-31 23:52:11.327000,2026-06-10 09:33:49.412000,87.375200,179.999400,681.238000,8.800000,1.000000,2910.000000,619.000000,62.55800,2.820000,358.180000,NaN,NaN,NaN,2025.000000,12.000000,31.000000,23.000000,1.000000
std,NaN,NaN,32.129209,128.847019,123.636465,0.607440,0.077126,91.623798,34.239985,4.39120,0.257163,61.848718,NaN,NaN,NaN,1.444235,3.466340,8.886061,6.997628,0.045114


In [84]:
num_cols = ['magError', 'depthError', 'magNst']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

    df['magError'] = df['magError'].fillna(0)
    df['depthError'] = df['depthError'].fillna(0)
    df['magNst'] = df['magNst'].fillna(0)

In [85]:
# Duplicate Remove

df.drop_duplicates(inplace=True)

In [86]:
# Save the clean dataset
df.to_csv("cleaned earthquake data.csv", index=False)

In [87]:
df.isnull().sum()
df["type"].value_counts()

,count
type,
earthquake,103628
mining explosion,770
ice quake,13
volcanic eruption,12
other event,5
landslide,4
experimental explosion,3
mine collapse,3
quarry blast,2


In [88]:
df["type"].value_counts()

,count
type,
earthquake,103628
mining explosion,770
ice quake,13
volcanic eruption,12
other event,5
landslide,4
experimental explosion,3
mine collapse,3
quarry blast,2


In [89]:
df["depth_category"].value_counts()

,count
depth_category,
Shallow,77976
Intermediate,20271
Deep,6194


MY SQL


In [90]:
import sqlite3

conn = sqlite3.connect("project1.db")

cursor = conn.cursor()

create_table_query = """
CREATE TABLE IF NOT EXISTS prj (id TEXT,time TIMESTAMP,
    updated TIMESTAMP,latitude REAL,longitude REAL,depth_km REAL,mag REAL,magType TEXT,place TEXT,status TEXT,
    tsunami INTEGER,sig INTEGER,net TEXT,nst REAL,dmin REAL,rms REAL,gap REAL,magError REAL,depthError REAL,
    magNst REAL,locationSource TEXT,magSource TEXT,types TEXT,ids TEXT,sources TEXT,type TEXT,country TEXT,
    year INTEGER,month INTEGER,day INTEGER,hour INTEGER,day_of_week TEXT,depth_category TEXT,magnitude_category TEXT,
    strong_eq INTEGER)"""

cursor.execute(create_table_query)

conn.commit()

print("Table created successfully")


Table created successfully


In [91]:
df.to_sql("prj", conn, if_exists="replace", index=False)

104441

In [92]:
#Table Checking

query = """
SELECT name
FROM sqlite_master
WHERE type='table'
"""

pd.read_sql(query, conn)

,name
0,prj


In [93]:
import sqlite3

conn = sqlite3.connect("project1.db")

cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS earthqu")

conn.commit()

print("Table deleted successfully")

Table deleted successfully


In [94]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table'
"""

pd.read_sql(query, conn)

,name
0,prj


Top 10 strongest earthquakes

In [95]:
query = "SELECT *FROM prj ORDER BY mag DESC LIMIT 10"
pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,us6000qw60,2025-07-29 23:24:52.483000,2026-05-29 17:29:47.842000,52.4948,160.2395,35.000,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,earthquake,Russia Earthquake,2025,7,29,23,Tuesday,Shallow,Great,1
1,ak0219neiszm,2021-07-29 06:15:49.188000,2026-04-15 18:42:11.216000,55.3635,-157.8876,35.000,8.2,mww,"2021 Chignik, Alaska Earthquake",reviewed,...,earthquake,Alaska Earthquake,2021,7,29,6,Thursday,Shallow,Great,1
2,us7000dflf,2021-03-04 19:28:33.178000,2025-12-22 18:58:46.851000,-29.7228,-177.2794,28.930,8.1,mww,"2021 Kermadec Islands, New Zealand Earthquake",reviewed,...,earthquake,New Zealand Earthquake,2021,3,4,19,Thursday,Shallow,Great,1
3,us6000f53e,2021-08-12 18:35:17.231000,2025-12-22 22:13:38.221000,-58.3753,-25.2637,22.790,8.1,mww,2021 South Sandwich Islands Earthquake,reviewed,...,earthquake,Unknown,2021,8,12,18,Thursday,Shallow,Great,1
4,us6000jllz,2023-02-06 01:17:34.342000,2026-04-13 23:32:30.032000,37.2256,37.0143,10.000,7.8,mww,"Pazarcik earthquake, Kahramanmaras earthquake ...",reviewed,...,earthquake,Kahramanmaras earthquake sequence,2023,2,6,1,Monday,Shallow,Major,1
5,us7000qx2g,2025-09-18 18:58:14.939000,2025-12-06 16:41:30.345000,53.1426,160.7206,27.000,7.8,mww,"140 km E of Petropavlovsk-Kamchatsky, Russia",reviewed,...,earthquake,Russia,2025,9,18,18,Thursday,Shallow,Major,1
6,us6000dg77,2021-02-10 13:19:55.530000,2025-12-22 18:54:06.521000,-23.0511,171.6566,10.000,7.7,mww,southeast of the Loyalty Islands,reviewed,...,earthquake,Unknown,2021,2,10,13,Wednesday,Shallow,Major,1
7,us6000kd0n,2023-05-19 02:57:03.172000,2025-08-15 23:01:19.104000,-23.2063,170.7423,18.053,7.7,mww,southeast of the Loyalty Islands,reviewed,...,earthquake,Unknown,2023,5,19,2,Friday,Shallow,Major,1
8,us7000pn9s,2025-03-28 06:20:52.715000,2026-05-15 09:12:34.190000,22.0110,95.9363,10.000,7.7,mww,"2025 Mandalay, Burma (Myanmar) Earthquake",reviewed,...,earthquake,Burma (Myanmar) Earthquake,2025,3,28,6,Friday,Shallow,Major,1
9,us7000i9bw,2022-09-19 18:05:08.217000,2025-12-10 03:40:12.288000,18.4552,-102.9561,26.943,7.6,mww,"35 km SSW of Aguililla, Mexico",reviewed,...,earthquake,Mexico,2022,9,19,18,Monday,Shallow,Major,1


Top 10 deepest earthquakes

In [96]:
query = """SELECT *
FROM prj
ORDER BY depth_km DESC
LIMIT 10"""

pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,us6000k2db,2023-04-01 18:09:17.114000,2023-06-03 22:06:21.040000,-13.1462,169.3241,681.238,4.0,mb,"208 km ENE of Sola, Vanuatu",reviewed,...,earthquake,Vanuatu,2023,4,1,18,Saturday,Deep,Light,0
1,us7000kxdn,2023-09-18 15:35:27.270000,2023-12-02 23:01:10.040000,-15.1098,171.6867,675.265,4.2,mb,Vanuatu region,reviewed,...,earthquake,Unknown,2023,9,18,15,Monday,Deep,Light,0
2,us6000mivr,2024-03-08 22:42:23.713000,2024-05-15 17:35:02.040000,-19.3114,-177.8158,671.043,4.2,mb,Fiji region,reviewed,...,earthquake,Unknown,2024,3,8,22,Friday,Deep,Light,0
3,us6000rk66,2025-10-29 17:07:34.157000,2026-01-27 17:59:01.040000,-19.0745,-179.0467,669.556,4.8,mb,"205 km ESE of Levuka, Fiji",reviewed,...,earthquake,Fiji,2025,10,29,17,Wednesday,Deep,Light,0
4,us6000f2w3,2021-08-01 22:12:46.340000,2021-10-14 01:08:25.040000,-20.0989,-179.0470,669.460,4.0,mb,"283 km SE of Levuka, Fiji",reviewed,...,earthquake,Fiji,2021,8,1,22,Sunday,Deep,Light,0
5,us7000q1jk,2025-05-10 02:03:16.756000,2025-07-24 21:30:18.040000,-17.8425,-177.8703,667.237,4.2,mb,"299 km E of Levuka, Fiji",reviewed,...,earthquake,Fiji,2025,5,10,2,Saturday,Deep,Light,0
6,us6000dhfx,2021-02-13 16:26:42.487000,2021-04-23 21:04:30.040000,-24.2149,179.7615,664.740,4.5,mb,south of the Fiji Islands,reviewed,...,earthquake,Unknown,2021,2,13,16,Saturday,Deep,Light,0
7,us7000he64,2022-06-01 18:41:01.389000,2022-08-06 22:13:14.040000,-16.9726,-178.0768,664.700,4.3,mb,"279 km ESE of Labasa, Fiji",reviewed,...,earthquake,Fiji,2022,6,1,18,Wednesday,Deep,Light,0
8,us6000m2wg,2023-12-31 02:19:26.839000,2024-03-02 21:39:37.040000,-13.0028,168.4651,660.826,4.1,mb,"138 km NE of Sola, Vanuatu",reviewed,...,earthquake,Vanuatu,2023,12,31,2,Sunday,Deep,Light,0
9,us7000ingi,2022-11-09 09:51:04.068000,2023-01-21 19:27:16.040000,-26.0901,178.3427,660.000,7.0,mwb,south of the Fiji Islands,reviewed,...,earthquake,Unknown,2022,11,9,9,Wednesday,Deep,Major,1


Shallow (<50km) and mag > 7.5

In [97]:
query = """SELECT *
FROM prj
WHERE depth_km < 50 AND mag > 7.5"""

pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,us6000dg77,2021-02-10 13:19:55.530000,2025-12-22 18:54:06.521000,-23.0511,171.6566,10.000,7.7,mww,southeast of the Loyalty Islands,reviewed,...,earthquake,Unknown,2021,2,10,13,Wednesday,Shallow,Major,1
1,us7000dflf,2021-03-04 19:28:33.178000,2025-12-22 18:58:46.851000,-29.7228,-177.2794,28.930,8.1,mww,"2021 Kermadec Islands, New Zealand Earthquake",reviewed,...,earthquake,New Zealand Earthquake,2021,3,4,19,Thursday,Shallow,Great,1
2,ak0219neiszm,2021-07-29 06:15:49.188000,2026-04-15 18:42:11.216000,55.3635,-157.8876,35.000,8.2,mww,"2021 Chignik, Alaska Earthquake",reviewed,...,earthquake,Alaska Earthquake,2021,7,29,6,Thursday,Shallow,Great,1
3,us6000f53e,2021-08-12 18:35:17.231000,2025-12-22 22:13:38.221000,-58.3753,-25.2637,22.790,8.1,mww,2021 South Sandwich Islands Earthquake,reviewed,...,earthquake,Unknown,2021,8,12,18,Thursday,Shallow,Great,1
4,us7000i9bw,2022-09-19 18:05:08.217000,2025-12-10 03:40:12.288000,18.4552,-102.9561,26.943,7.6,mww,"35 km SSW of Aguililla, Mexico",reviewed,...,earthquake,Mexico,2022,9,19,18,Monday,Shallow,Major,1
5,us6000jllz,2023-02-06 01:17:34.342000,2026-04-13 23:32:30.032000,37.2256,37.0143,10.000,7.8,mww,"Pazarcik earthquake, Kahramanmaras earthquake ...",reviewed,...,earthquake,Kahramanmaras earthquake sequence,2023,2,6,1,Monday,Shallow,Major,1
6,us6000kd0n,2023-05-19 02:57:03.172000,2025-08-15 23:01:19.104000,-23.2063,170.7423,18.053,7.7,mww,southeast of the Loyalty Islands,reviewed,...,earthquake,Unknown,2023,5,19,2,Friday,Shallow,Major,1
7,us7000lff4,2023-12-02 14:37:04.454000,2026-05-04 17:53:24.474000,8.5266,126.4161,40.000,7.6,mww,"19 km E of Gamut, Philippines",reviewed,...,earthquake,Philippines,2023,12,2,14,Saturday,Shallow,Major,1
8,us7000pcdl,2025-02-08 23:23:14.697000,2025-04-26 15:51:49.572000,17.6506,-82.3950,14.326,7.6,mww,"210 km SSW of George Town, Cayman Islands",reviewed,...,earthquake,Cayman Islands,2025,2,8,23,Saturday,Shallow,Major,1
9,us7000pn9s,2025-03-28 06:20:52.715000,2026-05-15 09:12:34.190000,22.0110,95.9363,10.000,7.7,mww,"2025 Mandalay, Burma (Myanmar) Earthquake",reviewed,...,earthquake,Burma (Myanmar) Earthquake,2025,3,28,6,Friday,Shallow,Major,1


Average depth per country

In [98]:
query = """SELECT country, AVG(depth_km) AS avg_depth
FROM prj
GROUP BY country"""
pd.read_sql(query, conn)

,country,avg_depth
0,AK,90.223333
1,Afghanistan,127.947718
2,Alabama,2.505000
3,Alaska,42.293250
4,Alaska Earthquake,32.666667
...,...,...
223,Washington,22.902625
224,Wyoming,0.462635
225,Yemen,10.000000
226,Zambia,10.219182


Average magnitude per magType

In [99]:
query = """SELECT magType, AVG(mag) AS avg_mag
FROM prj
GROUP BY magType"""

pd.read_sql(query, conn)


,magType,avg_mag
0,mb,4.425133
1,mb_lg,3.208696
2,md,3.447550
3,mh,4.100000
4,ml,3.274783
5,ml(texnet),3.367442
6,mlg,3.300000
7,mlr,3.480000
8,mlv,3.469288
9,ms_20,5.800000


Year with most earthquakes

In [100]:
query = """SELECT year, COUNT(*) AS total
FROM prj
GROUP BY year
ORDER BY total DESC
LIMIT 1"""

pd.read_sql(query, conn)

,year,total
0,2025,22818


In [101]:
#Month with most earthquakes

query = """SELECT month, COUNT(*) AS total
FROM prj
GROUP BY month
ORDER BY total DESC
LIMIT 1"""

pd.read_sql(query, conn)

,month,total
0,8,9857


In [102]:
#Day of week analysis

query = """SELECT day_of_week, COUNT(*) AS total
FROM prj
GROUP BY day_of_week
ORDER BY total DESC"""

pd.read_sql(query, conn)

,day_of_week,total
0,Friday,15100
1,Tuesday,15076
2,Sunday,15068
3,Wednesday,14933
4,Saturday,14856
5,Monday,14770
6,Thursday,14638


In [103]:
#Earthquakes per hour

query = """
SELECT strftime('%H', time) AS hour, COUNT(*) AS total
FROM prj
GROUP BY hour
ORDER BY total DESC
"""
pd.read_sql(query, conn)

,hour,total
0,03,4683
1,21,4648
2,04,4647
3,01,4596
4,22,4512
5,20,4458
6,02,4430
7,19,4428
8,18,4400
9,07,4345


In [104]:
# Most active network

query = """SELECT net, COUNT(*) AS total
FROM prj
GROUP BY net
ORDER BY total DESC
LIMIT 1"""

pd.read_sql(query, conn)

,net,total
0,us,91604


In [105]:
#Reviewed vs automatic
query = """SELECT status, COUNT(*) AS total
FROM prj
GROUP BY status"""

pd.read_sql(query, conn)

,status,total
0,automatic,149
1,reviewed,104292


In [106]:
#Event type count

query = """SELECT type, COUNT(*) AS total
FROM prj
GROUP BY type"""

pd.read_sql(query, conn)

,type,total
0,earthquake,103628
1,experimental explosion,3
2,explosion,1
3,ice quake,13
4,landslide,4
5,mine collapse,3
6,mining explosion,770
7,other event,5
8,quarry blast,2
9,volcanic eruption,12


In [107]:
#Data types count
query = """SELECT types, COUNT(*) AS total
FROM prj
GROUP BY types"""

pd.read_sql(query, conn)

,types,total
0,",associate,disassociate,dyfi,ground-failure,im...",1
1,",associate,disassociate,dyfi,losspager,origin,...",1
2,",associate,disassociate,dyfi,moment-tensor,ori...",1
3,",associate,disassociate,dyfi,nearby-cities,ori...",1
4,",associate,disassociate,dyfi,origin,phase-data,",2
...,...,...
547,",oaf,origin,shake-alert,",1
548,",origin,",15
549,",origin,phase-data,",79557
550,",origin,phase-data,shakemap,",1639


In [108]:
#Avg RMS & GAP per country

query = """SELECT country,
AVG(rms) AS avg_rms,
AVG(gap) AS avg_gap
FROM prj
GROUP BY country"""

pd.read_sql(query, conn)

,country,avg_rms,avg_gap
0,AK,0.583333,144.000000
1,Afghanistan,0.722193,94.071658
2,Alabama,0.480000,94.000000
3,Alaska,0.620593,172.112977
4,Alaska Earthquake,0.796667,65.000000
...,...,...,...
223,Washington,0.285750,67.325000
224,Wyoming,0.619985,89.130368
225,Yemen,0.722754,117.942029
226,Zambia,0.710909,70.181818


In [109]:
#High station coverage

query = """SELECT *
FROM prj
WHERE nst > 50"""

pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,hv72334707,2021-01-29 19:10:06.900000,2021-04-16 19:02:21.040000,19.395833,-155.442833,8.130,3.24,ml,"21 km N of P?hala, Hawaii",reviewed,...,earthquake,Hawaii,2021,1,29,19,Friday,Shallow,Light,0
1,hv72334632,2021-01-29 16:33:43.180000,2021-04-16 19:02:20.040000,19.269167,-155.799667,9.160,3.24,ml,"21 km SSE of Honaunau-Napoopoo, Hawaii",reviewed,...,earthquake,Hawaii,2021,1,29,16,Friday,Shallow,Light,0
2,ci39768296,2021-01-26 07:48:47.420000,2022-01-13 05:23:15.619000,33.089667,-117.622833,13.770,3.14,ml,"26km WSW of Carlsbad, CA",reviewed,...,earthquake,CA,2021,1,26,7,Tuesday,Shallow,Light,0
3,ci39768064,2021-01-25 20:52:28.360000,2021-04-13 17:04:32.040000,32.738333,-115.823667,10.720,3.01,ml,"16km E of Ocotillo, CA",reviewed,...,earthquake,CA,2021,1,25,20,Monday,Shallow,Light,0
4,nc73514715,2021-01-22 00:48:11.350000,2021-11-17 20:07:32.380000,37.968667,-122.027000,15.440,3.01,ml,"1km W of Concord, CA",reviewed,...,earthquake,CA,2021,1,22,0,Friday,Shallow,Light,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22180,us7000reu0,2025-12-01 05:28:16.530000,2026-02-25 16:18:40.040000,-20.451800,-175.699700,238.900,4.40,mb,"89 km NNW of Houma, Tonga",reviewed,...,earthquake,Tonga,2025,12,1,5,Monday,Intermediate,Light,0
22181,us7000retu,2025-12-01 04:26:20.780000,2026-02-25 16:18:40.040000,27.585000,56.130800,10.000,4.40,mb,"46 km NNW of Bandar Abbas, Iran",reviewed,...,earthquake,Iran,2025,12,1,4,Monday,Shallow,Light,0
22182,ak2025xpgfse,2025-12-01 01:51:06.974000,2026-02-25 16:18:40.040000,59.138000,-150.640000,15.200,3.20,ml,"60 km SSE of Halibut Cove, Alaska",reviewed,...,earthquake,Alaska,2025,12,1,1,Monday,Shallow,Light,0
22183,us7000ret3,2025-12-01 01:41:43.796000,2026-02-25 16:18:40.040000,7.412700,127.070400,35.000,4.60,mb,"53 km E of Baculin, Philippines",reviewed,...,earthquake,Philippines,2025,12,1,1,Monday,Shallow,Light,0


In [110]:
#Tsunami count per year

query = """SELECT year, COUNT(*) AS tsunami_count
FROM prj
WHERE tsunami = 1
GROUP BY year"""

pd.read_sql(query, conn)

,year,tsunami_count
0,2021,114
1,2022,136
2,2023,119
3,2024,114
4,2025,142


In [111]:
#Alert distribution

query = """
SELECT magType, COUNT(*) AS total
FROM prj
GROUP BY magType
"""
pd.read_sql(query, conn)

,magType,total
0,mb,74318
1,mb_lg,115
2,md,5620
3,mh,1
4,ml,14857
5,ml(texnet),43
6,mlg,1
7,mlr,10
8,mlv,5
9,ms_20,1


In [112]:
#Top 5 countries by magnitude

query = """SELECT country, AVG(mag) AS avg_mag
FROM prj
GROUP BY country
ORDER BY avg_mag DESC
LIMIT 5"""

pd.read_sql(query, conn)

,country,avg_mag
0,Russia Earthquake,8.100000
1,New Zealand Earthquake,8.100000
2,Burma (Myanmar) Earthquake,7.700000
3,Kahramanmaras earthquake sequence,7.650000
4,Alaska Earthquake,7.566667


In [113]:
#Shallow + deep same month

query = """SELECT country, month
FROM prj
GROUP BY country, month
HAVING SUM(CASE WHEN depth_km < 50 THEN 1 ELSE 0 END) > 0
AND SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) > 0"""

pd.read_sql(query, conn)

,country,month
0,Afghanistan,10
1,Argentina,1
2,Argentina,2
3,Argentina,3
4,Argentina,4
...,...,...
191,Wallis and Futuna,8
192,Wallis and Futuna,9
193,Wallis and Futuna,10
194,Wallis and Futuna,11


In [114]:
#Year-over-year growth

query = """SELECT year,
COUNT(*) AS total
FROM prj
GROUP BY year
ORDER BY year"""

pd.read_sql(query, conn)

,year,total
0,2021,21926
1,2022,20208
2,2023,20830
3,2024,18659
4,2025,22818


In [115]:
#Top active regions

query = """SELECT country,
COUNT(*) AS freq,
AVG(mag) AS avg_mag
FROM prj
GROUP BY country
ORDER BY freq DESC, avg_mag DESC
LIMIT 3"""

pd.read_sql(query, conn)

,country,freq,avg_mag
0,Unknown,19726,4.554582
1,Alaska,12396,3.466845
2,Indonesia,7840,4.494554


In [116]:
#Equator zone earthquakes

query = """SELECT country, AVG(depth_km) AS avg_depth
FROM prj
WHERE latitude BETWEEN -5 AND 5
GROUP BY country"""

pd.read_sql(query, conn)

,country,avg_depth
0,Brazil,10.556000
1,Burundi,10.000000
2,Colombia,39.175912
3,Congo-Uganda,14.775000
4,Democratic Republic of the Congo,9.914739
5,Ecuador,60.633676
6,Ecuador region,10.000000
7,Ethiopia,10.000000
8,Gabon,9.818182
9,Guyana,10.351429


In [117]:
#Shallow vs deep ratio
query = """SELECT country,
SUM(CASE WHEN depth_km < 50 THEN 1 ELSE 0 END) /
SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) AS ratio
FROM prj
GROUP BY country"""

pd.read_sql(query, conn)

,country,ratio
0,AK,NaN
1,Afghanistan,220.0
2,Alabama,NaN
3,Alaska,NaN
4,Alaska Earthquake,NaN
...,...,...
223,Washington,NaN
224,Wyoming,NaN
225,Yemen,NaN
226,Zambia,NaN


In [118]:
#Tsunami vs magnitude
query = """SELECT tsunami, AVG(mag) AS avg_mag
FROM prj
GROUP BY tsunami"""

pd.read_sql(query, conn)

,tsunami,avg_mag
0,0,4.25343
1,1,5.41728


In [119]:
#High risk events (gap + rms)
query = """SELECT *
FROM prj
ORDER BY (gap + rms) DESC
LIMIT 10"""

pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,nn00897599,2025-05-14 12:02:09.568000,2025-07-25 17:06:15.040000,41.2170,-116.6817,0.00,3.00,ml,"60 km NE of Valmy, Nevada",reviewed,...,earthquake,Nevada,2025,5,14,12,Wednesday,Shallow,Light,0
1,pr2024051000,2024-02-20 05:23:09.140000,2024-05-02 15:50:10.040000,18.6156,-63.6768,30.00,3.58,md,"77 km NW of Sandy Ground Village, Anguilla",reviewed,...,earthquake,Anguilla,2024,2,20,5,Tuesday,Shallow,Light,0
2,us7000denb,2021-02-14 19:26:12.202000,2021-04-23 21:04:37.040000,51.2911,-179.2451,35.00,3.30,ml,"Andreanof Islands, Aleutian Islands, Alaska",reviewed,...,earthquake,Alaska,2021,2,14,19,Sunday,Shallow,Light,0
3,pr2022227000,2022-08-15 01:27:45.500000,2022-08-15 02:29:01.489000,18.4843,-64.2523,14.00,3.63,md,"59 km ENE of Cruz Bay, U.S. Virgin Islands",reviewed,...,earthquake,U.S. Virgin Islands,2022,8,15,1,Monday,Shallow,Light,0
4,pr71489653,2025-07-18 23:19:06.230000,2025-07-19 17:53:06.150000,17.9070,-68.7190,46.94,3.13,md,"53 km SSW of Boca de Yuma, Dominican Republic",reviewed,...,earthquake,Dominican Republic,2025,7,18,23,Friday,Shallow,Light,0
5,pr2022289000,2022-10-16 20:43:51.670000,2022-10-16 22:18:14.728000,18.3415,-64.0490,44.00,3.48,md,"78 km E of Cruz Bay, U.S. Virgin Islands",reviewed,...,earthquake,U.S. Virgin Islands,2022,10,16,20,Sunday,Shallow,Light,0
6,pr2021003002,2021-01-03 01:55:58.430000,2021-03-13 22:58:56.040000,18.0363,-63.9581,32.00,3.53,md,"87 km WNW of The Bottom, Bonaire, Saint Eustat...",reviewed,...,earthquake,Saint Eustatius and Saba,2021,1,3,1,Sunday,Shallow,Light,0
7,pr2022311006,2022-11-07 08:05:45.780000,2022-11-07 15:22:33.753000,18.7130,-64.4161,52.00,3.13,md,"58 km NE of Cruz Bay, U.S. Virgin Islands",reviewed,...,earthquake,U.S. Virgin Islands,2022,11,7,8,Monday,Shallow,Light,0
8,pr2021028009,2021-01-28 05:57:59.030000,2021-04-13 17:06:06.040000,17.7576,-64.1211,69.00,3.70,md,"66 km E of Saint Croix, U.S. Virgin Islands",reviewed,...,earthquake,U.S. Virgin Islands,2021,1,28,5,Thursday,Shallow,Light,0
9,pr2021037008,2021-02-06 08:50:34.790000,2021-02-06 09:51:06.282000,19.1478,-64.7251,38.00,3.38,md,"90 km N of Cruz Bay, U.S. Virgin Islands",reviewed,...,earthquake,U.S. Virgin Islands,2021,2,6,8,Saturday,Shallow,Light,0


In [120]:
#Deep earthquakes count

query = """SELECT country, COUNT(*) AS deep_count
FROM prj
WHERE depth_km > 300
GROUP BY country"""

pd.read_sql(query, conn)

,country,deep_count
0,Afghanistan,1
1,Argentina,34
2,Bolivia,19
3,Brazil,14
4,China,1
5,Fiji,1114
6,Guam,1
7,Indonesia,260
8,Italy,11
9,Japan,116


In [121]:
query = "SELECT COUNT(*) FROM prj"
pd.read_sql(query, conn)

,COUNT(*)
0,104441


In [122]:
query = "SELECT * FROM prj LIMIT 5"
pd.read_sql(query, conn)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,country,year,month,day,hour,day_of_week,depth_category,magnitude_category,strong_eq
0,us6000ddi8,2021-01-31 23:20:49.923000,2021-04-16 19:02:44.040000,-31.7493,-68.9337,17.27,4.7,mwr,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,...,earthquake,Argentina,2021,1,31,23,Sunday,Shallow,Light,0
1,us6000dev6,2021-01-31 23:08:17.161000,2021-04-16 19:03:47.040000,-15.4902,-177.2052,426.71,4.1,mb,Fiji region,reviewed,...,earthquake,Unknown,2021,1,31,23,Sunday,Deep,Light,0
2,us6000dev5,2021-01-31 22:54:19.760000,2021-04-16 19:03:47.040000,19.7529,121.3159,46.73,4.7,mb,"103 km SW of Basco, Philippines",reviewed,...,earthquake,Philippines,2021,1,31,22,Sunday,Shallow,Light,0
3,us6000ddhs,2021-01-31 22:06:00.832000,2021-04-16 19:02:43.040000,28.1524,57.2570,10.00,4.9,mb,"114 km N of M?n?b, Iran",reviewed,...,earthquake,Iran,2021,1,31,22,Sunday,Shallow,Light,0
4,us6000dev4,2021-01-31 21:51:14.016000,2021-04-16 19:03:46.040000,71.3212,-3.7578,10.00,4.0,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,...,earthquake,Svalbard and Jan Mayen,2021,1,31,21,Sunday,Shallow,Light,0
